# 价格预测实验主入口

## 工作流

1. 运行 `data/data_pipeline.ipynb` 中的 `LOBDataPipeline`，构建统一特征 + 价格标签；
2. 运行 `baselines/baselines.ipynb` 中的小模型（LightGBM / LSTM / TCN）；
3. 运行 `models/ts_models.ipynb` 中的时序大模型（Chronos / TimesFM / PatchTST / Informer / Lag-Llama / DeepLOB）；
4. 使用 `evaluation/evaluation.ipynb` 中的函数进行统一评估与回测。

> **使用方法**：在本 notebook 的 Jupyter 内核中，先依次运行上述四个 notebook 的核心 cell（使 class/function 进入内存），然后运行本 notebook 的 cell。
>
> 或者，你可以把各 notebook 的核心代码导出为 `.py` 模块后用 `import` 导入。

In [ ]:
# 如果你选择在同一个内核中运行，需要先执行以下四个 notebook 的核心 cell：
#   1. data/data_pipeline.ipynb    -> 定义 LOBDataPipeline
#   2. baselines/baselines.ipynb   -> 定义 LightGBMBaseline / LSTMBaseline / TCNBaseline
#   3. models/ts_models.ipynb      -> 定义各 Wrapper
#   4. evaluation/evaluation.ipynb -> 定义 regression_metrics 等
#
# 以下代码假设上述类/函数已在内存中可用。

import numpy as np

# --------------------------------------------------
# 配置区（可改为从 configs/*.yaml 加载）
# --------------------------------------------------
DATA_DIR    = "data/processed"    # 存放 {ticker}_1min.csv
TICKER      = "1319"             # 示例股票
LOOKBACK    = 60                 # 历史窗口（分钟）
HORIZON     = 30                 # 预测未来（分钟）
TRAIN_RATIO = 0.8                # 简单按比例切训练/测试

FEATURE_COLS = [
    "mid_price", "spread", "order_imbalance", "log_return",
    "vol_5", "vol_10", "vol_20",
    "vol_ma_5", "vol_ma_10", "vol_ma_20", "vol_spike",
]

In [ ]:
# --------------------------------------------------
# Step 1: 数据管道
# --------------------------------------------------
pipeline = LOBDataPipeline(data_dir=DATA_DIR, lookback=LOOKBACK, horizon=HORIZON)

df_raw = pipeline.load_raw(TICKER)
df_feat = pipeline.engineer_features(df_raw)
df_labeled = pipeline.build_price_labels(df_feat)

# 过滤可用特征列
feat_cols = [c for c in FEATURE_COLS if c in df_labeled.columns]
print(f"使用特征 ({len(feat_cols)}): {feat_cols}")

X_seq, X_flat, y = pipeline.build_sequences(df_labeled, feat_cols)
print(f"X_seq: {X_seq.shape}  X_flat: {X_flat.shape}  y: {y.shape}")

In [ ]:
# --------------------------------------------------
# Step 2: 训练/测试切分（按时间顺序）
# --------------------------------------------------
split = int(len(y) * TRAIN_RATIO)

X_seq_train, X_seq_test = X_seq[:split], X_seq[split:]
X_flat_train, X_flat_test = X_flat[:split], X_flat[split:]
y_train, y_test = y[:split], y[split:]

# 当前价格（用于方向准确率 & 回测）：取特征中 mid_price 列的最后一个时间步
mid_idx = feat_cols.index("mid_price") if "mid_price" in feat_cols else 0
y_current_test = X_seq_test[:, -1, mid_idx]

print(f"训练集: {len(y_train)}  测试集: {len(y_test)}")

In [ ]:
# --------------------------------------------------
# Step 3: 运行小模型基线
# --------------------------------------------------
results = []

# --- LightGBM ---
lgbm = LightGBMBaseline()
lgbm.fit(X_flat_train, y_train)
y_pred_lgbm = lgbm.predict(X_flat_test)
m = regression_metrics(y_test, y_pred_lgbm)
m["model"] = "LightGBM"
m["train_time"] = round(lgbm.train_time, 2)
m["infer_time"] = round(lgbm.infer_time, 4)
m["dir_acc"] = round(direction_accuracy(y_test, y_pred_lgbm, y_current_test), 4)
results.append(m)
print("LightGBM done:", m)

# --- LSTM ---
lstm = LSTMBaseline(input_dim=len(feat_cols), hidden_dim=64, epochs=30)
lstm.fit(X_seq_train, y_train)
y_pred_lstm = lstm.predict(X_seq_test)
m = regression_metrics(y_test, y_pred_lstm)
m["model"] = "LSTM"
m["train_time"] = round(lstm.train_time, 2)
m["infer_time"] = round(lstm.infer_time, 4)
m["dir_acc"] = round(direction_accuracy(y_test, y_pred_lstm, y_current_test), 4)
results.append(m)
print("LSTM done:", m)

# --- TCN ---
tcn = TCNBaseline(input_dim=len(feat_cols), epochs=30)
tcn.fit(X_seq_train, y_train)
y_pred_tcn = tcn.predict(X_seq_test)
m = regression_metrics(y_test, y_pred_tcn)
m["model"] = "TCN"
m["train_time"] = round(tcn.train_time, 2)
m["infer_time"] = round(tcn.infer_time, 4)
m["dir_acc"] = round(direction_accuracy(y_test, y_pred_tcn, y_current_test), 4)
results.append(m)
print("TCN done:", m)

In [ ]:
# --------------------------------------------------
# Step 4: 对比表
# --------------------------------------------------
print("\n========== 模型对比 ==========")
df_results = print_comparison_table(results)

In [ ]:
# --------------------------------------------------
# Step 5: 简易回测（以 LightGBM 为例）
# --------------------------------------------------
import matplotlib.pyplot as plt

bt = simple_backtest(y_test, y_pred_lgbm, y_current_test)
print("LightGBM 回测结果:")
for k, v in bt.items():
    if k != "cumulative_pnl":
        print(f"  {k}: {v:.4f}")

plt.figure(figsize=(12, 4))
plt.plot(bt["cumulative_pnl"])
plt.title("LightGBM Cumulative PnL")
plt.xlabel("Time Step")
plt.ylabel("Cumulative PnL")
plt.grid(True)
plt.tight_layout()
plt.show()

## TODO

- [ ] 在 `models/ts_models.ipynb` 中补全 Chronos / TimesFM / PatchTST / Informer / Lag-Llama / DeepLOB 的 predict / fit 逻辑；
- [ ] 在上面 Step 3 中增加大模型的调用，加入 `results` 对比表；
- [ ] 从 `configs/*.yaml` 加载超参，替代硬编码；
- [ ] 多股票循环实验 + 聚合指标。